# HMS v11 — is the boundary cue in the representation, and does the fill type matter?

Two reviewer-driven experiments on 200 s windows (same data, folds and recipe as before):

**A. Embedding probe (inference only, ~20 min).** For the trained 50 s, original 200 s and boundary-agnostic (BA) 200 s models of Mamba and GRU (seed 42), the pooled representation that feeds the classifier head is extracted for every validation EEG. A linear probe (logistic regression, patient-grouped 5-fold CV inside each fold model's validation set) predicts whether the recording has **no extra real context** (a short excerpt). A 50 s model cannot see the boundary, so its probe AUC measures what the EEG and the label mix alone reveal about the annotation round; if the original 200 s model's AUC is far above that and the BA model's is not, the cue is shown to be encoded, not only inferred from outputs. Probes are also run inside the ≥10-vote stratum and on the true label vector (label-mediated ceiling).

**B. Fill-type ablation for BA training (~3.7 h).** BA training mirror-fills outside the recording. Two alternatives are trained for Mamba and GRU, otherwise identical (random truncation p = 0.5, no reweighting, seed 42):
- `ba_splice` — EEG from a different, randomly chosen recording fills the window outside the real signal;
- `ba_noise` — band-limited (0.5–40 Hz) Gaussian noise scaled to the recording's per-channel SD.
Each is evaluated like the mirror version: full view (`ba`) and central 50 s only (`trunc50`), giving the matched-input asymmetry against the 50 s model and the real-context effect.

**Inputs (GPU T4, Internet off):** competition data; outputs of `notebook86c832275b` (Version 5; 50 s/200 s predictions and the GRU 200 s checkpoints), `notebookd13836b6e5` (50 s and original 200 s checkpoints), `notebookbb6726eb71` (Mamba BA checkpoints and predictions), `notebook22e15f591d` (GRU BA predictions) and the dataset `hms-v6-ckpts` (GRU BA checkpoints). To resume, attach this notebook's own output too. **Save Version → Save & Run All.**

In [1]:
import os, glob, json, math, time, random, shutil
import numpy as np
import pandas as pd

CFG = dict(
    DATA_ROOT=None,                       # None = auto-detect under /kaggle/input
    CACHE_DIR="/kaggle/working/cache",    # replaced in the v2 settings cell below
    RESULTS_DIR="/kaggle/working/results",
    FS=100,                               # target sampling rate (Hz)
    CTX_SEC=50,
    CACHE_CTX_SEC=100,
    JITTER_SEC=1.0,                       # train-time random shift of the window centre
    PATCH=25,                             # samples per token (0.25 s at 100 Hz)
    N_FOLDS=5, FOLDS=[0], SEED=42,
    MODELS=["mamba", "transformer", "gru", "cnn"],
    D_MODEL=128, N_LAYERS=4, D_STATE=64, HEADDIM=32, EXPAND=2, CHUNK=64, N_HEADS_ATTN=4,
    EPOCHS=15, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0,
    TRAIN_MIN_VOTES=0,
    HIGH_VOTE=10,                         # threshold for the high-agreement evaluation subset
    NUM_WORKERS=4, AMP=True,
    USE_OFFICIAL_MAMBA2=False,
    RUN_BENCHMARK=True,
    BENCH_CTX=[50, 100, 200, 400, 600, 1200],
)

In [2]:
import pyarrow.parquet as pq
import multiprocessing as mp
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
MP = mp.get_context("fork")   # Kaggle is Linux; fork shares globals with workers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "")

def find_root():
    if CFG["DATA_ROOT"]:
        return CFG["DATA_ROOT"]
    for pat in ["/kaggle/input/*/train.csv", "/kaggle/input/*/*/train.csv"]:
        for h in glob.glob(pat):
            if os.path.isdir(os.path.join(os.path.dirname(h), "train_eegs")):
                return os.path.dirname(h)
    raise FileNotFoundError("Attach the competition data: Add Input → Competitions → HMS - Harmful Brain Activity Classification")

ROOT = find_root()
print("data:", ROOT)

device: cuda Tesla T4
data: /kaggle/input/competitions/hms-harmful-brain-activity-classification


## Labels and patient-level folds

In [3]:
VOTES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
CLASSES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

df = pd.read_csv(f"{ROOT}/train.csv")
df["n_votes"] = df[VOTES].sum(axis=1)
P_COLS = [f"p_{c}" for c in CLASSES]
df[P_COLS] = df[VOTES].div(df["n_votes"], axis=0).to_numpy()
df["y"] = df[P_COLS].to_numpy().argmax(1)
df["center"] = df["eeg_label_offset_seconds"] + 25.0   # experts labelled the central 10 s of each 50 s window

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
df["fold"] = -1
for k, (_, va) in enumerate(sgkf.split(df, df["y"], groups=df["patient_id"])):
    df.loc[df.index[va], "fold"] = k
assert df.groupby("patient_id")["fold"].nunique().max() == 1, "patient leaked across folds"

print(f"rows={len(df):,}  eegs={df.eeg_id.nunique():,}  patients={df.patient_id.nunique():,}")
print(df.groupby("fold").agg(rows=("eeg_id", "size"), eegs=("eeg_id", "nunique"), patients=("patient_id", "nunique")))
print(df.drop_duplicates("eeg_id")["expert_consensus"].value_counts())

rows=106,800  eegs=17,089  patients=1,950
       rows  eegs  patients
fold                       
0     17085  2795       383
1     25502  3689       386
2     21174  3585       394
3     22141  3238       397
4     20898  3782       390
expert_consensus
Other      7196
Seizure    2785
LPD        2577
GRDA       1828
GPD        1787
LRDA        916
Name: count, dtype: int64


## Preprocessing
18-channel bipolar double banana + midline (identical to I-CARE's 18 bipolar pairs). Band-pass 0.5–40 Hz at 200 Hz, then every second sample (100 Hz; the 40 Hz low-pass prevents aliasing). Amplitudes stay in µV (no per-window z-scoring), because absolute voltage — suppression, high-amplitude discharges — is clinically informative.

In [4]:
BIPOLAR = [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1"),    # left temporal chain
           ("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2"),    # right temporal chain
           ("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1"),    # left parasagittal chain
           ("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2"),    # right parasagittal chain
           ("Fz", "Cz"), ("Cz", "Pz")]                                 # midline
FLIP_PERM = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11, 16, 17]   # left <-> right hemisphere
ELECTRODES = sorted({e for pair in BIPOLAR for e in pair})
N_CH = len(BIPOLAR)
SRC_FS = 200
SOS = butter(4, [0.5, 40.0], btype="band", fs=SRC_FS, output="sos")
CLIP_UV = 500.0

def load_bipolar(path):
    d = pd.read_parquet(path, columns=ELECTRODES)
    d = d.interpolate(limit_direction="both").fillna(0.0)
    a = d.to_numpy(np.float32)
    col = {c: i for i, c in enumerate(ELECTRODES)}
    x = np.stack([a[:, col[p]] - a[:, col[q]] for p, q in BIPOLAR], axis=1)
    x = sosfiltfilt(SOS, x, axis=0)
    return np.clip(x, -CLIP_UV, CLIP_UV).astype(np.float32)

def _nrows(eid):
    return pq.ParquetFile(f"{ROOT}/train_eegs/{eid}.parquet").metadata.num_rows

def plan_cache(df, half_sec):
    g = df.groupby("eeg_id")["center"].agg(["min", "max"]).reset_index()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        n = pool.map(_nrows, g["eeg_id"].tolist(), chunksize=64)
    g["n_src"] = n
    s0 = np.floor((g["min"] - half_sec) * SRC_FS).clip(lower=0).astype(np.int64)
    s0 -= s0 % 2
    s1 = np.minimum(g["n_src"], np.ceil((g["max"] + half_sec) * SRC_FS)).astype(np.int64)
    g["s0"], g["length"] = s0, ((s1 - s0 + 1) // 2).clip(lower=0)
    g["offset"] = np.concatenate([[0], np.cumsum(g["length"].to_numpy())[:-1]])
    return g[["eeg_id", "n_src", "s0", "length", "offset"]]

def _build_one(args):
    eid, s0, length = args
    x = load_bipolar(f"{ROOT}/train_eegs/{eid}.parquet")[s0::2][:length]
    out = np.zeros((length, N_CH), np.float16)
    out[:len(x)] = x
    return out

def build_cache(df):
    meta_path = f"{CFG['CACHE_DIR']}/meta.json"
    want = dict(cache_ctx=CFG["CACHE_CTX_SEC"], jitter=CFG["JITTER_SEC"], fs=CFG["FS"], n_eegs=int(df.eeg_id.nunique()))
    if os.path.exists(meta_path) and json.load(open(meta_path)) == want:
        print("cache found, reusing")
        return pd.read_csv(f"{CFG['CACHE_DIR']}/index.csv")
    half = CFG["CACHE_CTX_SEC"] / 2 + CFG["JITTER_SEC"]
    idx = plan_cache(df, half)
    total = int(idx["length"].sum())
    need = total * N_CH * 2
    free = shutil.disk_usage(CFG["CACHE_DIR"]).free
    print(f"cache: {total/CFG['FS']/3600:,.1f} h of EEG, {need/1e9:.2f} GB needed, {free/1e9:.1f} GB free")
    if need > 0.9 * free:
        raise RuntimeError("Not enough disk: lower CACHE_CTX_SEC or point CACHE_DIR at a larger disk")
    mm = np.lib.format.open_memmap(f"{CFG['CACHE_DIR']}/signals.npy", mode="w+", dtype=np.float16, shape=(total, N_CH))
    jobs = list(zip(idx["eeg_id"], idx["s0"], idx["length"]))
    t0 = time.time()
    with MP.Pool(CFG["NUM_WORKERS"]) as pool:
        for i, (x, off) in enumerate(zip(pool.imap(_build_one, jobs, chunksize=16), idx["offset"])):
            mm[off:off + len(x)] = x
            if i % 2000 == 0:
                print(f"  {i:,}/{len(jobs):,}  {time.time()-t0:.0f}s")
    mm.flush(); del mm
    idx.to_csv(f"{CFG['CACHE_DIR']}/index.csv", index=False)
    json.dump(want, open(meta_path, "w"))
    print(f"cache built in {time.time()-t0:.0f}s")
    return idx

## Dataset
Each item is a `CTX_SEC` window centred on the labelled 10 s. Positions outside the recording are zero-filled and flagged in a mask; the SSM treats masked steps as identity transitions (Δ = 0), the Transformer ignores them via key-padding, the readout never pools them.
Training samples one random sub-window per EEG per epoch, so long recordings with many overlapping sub-windows don't dominate. Validation uses the middle sub-window of each EEG (the competition test set also has one row per EEG).

In [5]:
class HMSDataset(Dataset):
    """Self-contained (no notebook globals in __getitem__), so it works with any worker start method."""
    def __init__(self, rows, ctx_sec, train):
        assert ctx_sec <= CFG["CACHE_CTX_SEC"], "CTX_SEC exceeds the cached context; rebuild the cache"
        self.eid = rows["eeg_id"].to_numpy()
        self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.y = rows["y"].to_numpy()
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs)
        assert self.n % CFG["PATCH"] == 0
        self.train, self.jitter, self.flip = train, CFG["JITTER_SEC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"
        self.mm = None

    def __len__(self):
        return len(self.eid)

    def __getitem__(self, i):
        if self.mm is None:                      # open lazily so each worker gets its own handle
            self.mm = np.load(self.path, mmap_mode="r")
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i]
        if self.train:
            c += np.random.uniform(-self.jitter, self.jitter)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        x = np.zeros((self.n, self.n_ch), np.float32)
        m = np.zeros(self.n, np.float32)
        a, b = max(i0, 0), min(i0 + self.n, length)
        if b > a:
            x[a - i0:b - i0] = self.mm[off + a:off + b]
            m[a - i0:b - i0] = 1.0
        x = x.T / 50.0                           # ~unit scale while keeping absolute amplitude
        if self.train:
            if np.random.rand() < 0.5:
                x = x[self.flip]
            x = x * np.random.uniform(0.9, 1.1)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(m), torch.from_numpy(self.p[i]), int(self.y[i])

class OnePerEEG(Sampler):
    def __init__(self, eids, seed):
        order = np.argsort(eids, kind="stable")
        _, starts = np.unique(eids[order], return_index=True)
        self.groups = np.split(order, starts[1:])
        self.seed, self.epoch = seed, 0
    def __len__(self):
        return len(self.groups)
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1
        idx = np.array([g[rng.integers(len(g))] for g in self.groups])
        rng.shuffle(idx)
        return iter(idx.tolist())

def middle_row_per_eeg(rows):
    rows = rows.sort_values(["eeg_id", "eeg_label_offset_seconds"])
    g = rows.groupby("eeg_id")
    return rows[g.cumcount().to_numpy() == (g["eeg_id"].transform("size") // 2).to_numpy()]

## Models
All models: `Stem` (conv + 0.25 s patching → tokens) → sequence mixer → readout (pool over the central 10 s ⊕ pool over the full context) → 6 logits. Only the mixer differs.

The SSM is a bidirectional Mamba-2 block. `ssd()` is the chunked state-space-duality algorithm (exact); `ssd_reference()` is the plain recurrence used to verify it. `Mamba2.step()` is the recurrent form used for streaming.

In [6]:
from torch.utils.checkpoint import checkpoint

def run_block(blk, *args):
    if CFG["GRAD_CKPT"] and blk.training and torch.is_grad_enabled():
        return checkpoint(blk, *args, use_reentrant=False)
    return blk(*args)

def segsum(x):
    """x: (..., T) log-decays -> (..., T, T) with out[i, j] = sum_{k=j+1..i} x_k for i >= j, else -inf."""
    T = x.size(-1)
    x = x[..., None].expand(*x.shape, T)
    low = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), -1)
    x = x.masked_fill(~low, 0)
    s = torch.cumsum(x, dim=-2)
    keep = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device), 0)
    return s.masked_fill(~keep, -torch.inf)

def ssd(X, A, B, C, chunk):
    """Selective SSM, chunked SSD form.  h_t = exp(A_t) h_{t-1} + X_t B_t^T,  y_t = h_t C_t
    X: (b, T, h, p) input already multiplied by dt;  A: (b, T, h) = dt * A (<= 0);
    B, C: (b, T, n), shared by all heads (Mamba-2 with ngroups = 1)."""
    b, T, h, p = X.shape
    pad = (-T) % chunk
    if pad:
        X = F.pad(X, (0, 0, 0, 0, 0, pad))
        B, C = F.pad(B, (0, 0, 0, pad)), F.pad(C, (0, 0, 0, pad))
        A = F.pad(A, (0, 0, 0, pad))
    c = (T + pad) // chunk
    X = X.reshape(b, c, chunk, h, p)
    B = B.reshape(b, c, chunk, -1)
    C = C.reshape(b, c, chunk, -1)
    A = A.reshape(b, c, chunk, h).permute(0, 3, 1, 2)            # (b, h, c, l)
    A_cs = torch.cumsum(A, dim=-1)
    L = torch.exp(segsum(A))                                     # (b, h, c, l, s)
    CB = torch.einsum("bcln,bcsn->bcls", C, B)
    Y_diag = torch.einsum("bcls,bhcls,bcshp->bclhp", CB, L, X)
    decay_states = torch.exp(A_cs[..., -1:] - A_cs)
    states = torch.einsum("bcln,bhcl,bclhp->bchpn", B, decay_states, X)
    states = torch.cat([torch.zeros_like(states[:, :1]), states], dim=1)
    decay_chunk = torch.exp(segsum(F.pad(A_cs[..., -1], (1, 0))))
    states = torch.einsum("bhzc,bchpn->bzhpn", decay_chunk, states)[:, :-1]
    Y_off = torch.einsum("bcln,bchpn,bhcl->bclhp", C, states, torch.exp(A_cs))
    return (Y_diag + Y_off).reshape(b, c * chunk, h, p)[:, :T]

def ssd_reference(X, A, B, C):
    b, T, h, p = X.shape
    state = X.new_zeros(b, h, p, B.size(-1))
    ys = []
    for t in range(T):
        state = torch.exp(A[:, t])[..., None, None] * state + X[:, t, :, :, None] * B[:, t, None, None, :]
        ys.append(torch.einsum("bhpn,bn->bhp", state, C[:, t]))
    return torch.stack(ys, 1)

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.w

class Mamba2(nn.Module):
    """Mamba-2 mixer (pure PyTorch). Masked steps get dt = 0: no decay, no input -> state passes through unchanged."""
    def __init__(self, d, d_state, headdim, expand, chunk, d_conv=4):
        super().__init__()
        self.di, self.n, self.p, self.chunk = expand * d, d_state, headdim, chunk
        self.h = self.di // headdim
        self.in_proj = nn.Linear(d, 2 * self.di + 2 * d_state + self.h, bias=False)
        cd = self.di + 2 * d_state
        self.conv = nn.Conv1d(cd, cd, d_conv, groups=cd, padding=d_conv - 1)
        dt = torch.exp(torch.rand(self.h) * (math.log(0.1) - math.log(1e-3)) + math.log(1e-3)).clamp(min=1e-4)
        self.dt_bias = nn.Parameter(dt + torch.log(-torch.expm1(-dt)))     # inverse softplus
        self.A_log = nn.Parameter(torch.log(torch.empty(self.h).uniform_(1, 16)))
        self.D = nn.Parameter(torch.ones(self.h))
        self.norm = RMSNorm(self.di)
        self.out_proj = nn.Linear(self.di, d, bias=False)

    def forward(self, u, m):
        b, T, _ = u.shape
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        xBC = xBC * m[..., None]
        xBC = F.silu(self.conv(xBC.transpose(1, 2))[..., :T].transpose(1, 2))
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        with torch.autocast(device_type=u.device.type, enabled=False):
            dt = F.softplus(dt.float() + self.dt_bias) * m[..., None]
            A = -torch.exp(self.A_log.float())
            x = x.float().reshape(b, T, self.h, self.p)
            y = ssd(x * dt[..., None], A * dt, Bm.float(), Cm.float(), self.chunk) + x * self.D[:, None]
        y = y.reshape(b, T, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z)))

    def init_state(self, b, device):
        cd = self.di + 2 * self.n
        return (torch.zeros(b, cd, self.conv.kernel_size[0], device=device),
                torch.zeros(b, self.h, self.p, self.n, device=device))

    def step(self, u, state):
        """One token, u: (b, d). Fixed-size state: conv window (b, cd, d_conv) + SSM state (b, h, p, n)."""
        conv_st, ssm_st = state
        b = u.size(0)
        z, xBC, dt = torch.split(self.in_proj(u), [self.di, self.di + 2 * self.n, self.h], dim=-1)
        conv_st = torch.roll(conv_st, -1, dims=-1)
        conv_st[..., -1] = xBC.float()
        xBC = F.silu((conv_st * self.conv.weight[:, 0, :]).sum(-1) + self.conv.bias)
        x, Bm, Cm = torch.split(xBC, [self.di, self.n, self.n], dim=-1)
        dt = F.softplus(dt.float() + self.dt_bias)
        dA = torch.exp(dt * -torch.exp(self.A_log.float()))
        x = x.float().reshape(b, self.h, self.p)
        ssm_st = ssm_st * dA[..., None, None] + (x * dt[..., None])[..., None] * Bm.float()[:, None, None, :]
        y = torch.einsum("bhpn,bn->bhp", ssm_st, Cm.float()) + x * self.D[:, None]
        y = y.reshape(b, self.di).to(u.dtype)
        return self.out_proj(self.norm(y * F.silu(z))), (conv_st, ssm_st)

class OfficialMamba2(nn.Module):
    def __init__(self, d, d_state, headdim, expand, chunk):
        super().__init__()
        from mamba_ssm import Mamba2 as M2
        self.m = M2(d_model=d, d_state=d_state, headdim=headdim, expand=expand, chunk_size=chunk)
    def forward(self, u, m):
        return self.m(u * m[..., None])

class BiSSMBlock(nn.Module):
    def __init__(self, d, bidir=True):
        super().__init__()
        cls = OfficialMamba2 if CFG["USE_OFFICIAL_MAMBA2"] else Mamba2
        mk = lambda: cls(d, CFG["D_STATE"], CFG["HEADDIM"], CFG["EXPAND"], CFG["CHUNK"])
        self.norm, self.fwd = nn.LayerNorm(d), mk()
        self.bwd = mk() if bidir else None
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = self.norm(h)
        mf = m.float()
        y = self.fwd(x, mf)
        if self.bwd is not None:
            y = y + self.bwd(x.flip(1), mf.flip(1)).flip(1)
        return h + self.drop(y)

class SSMMixer(nn.Module):
    """bidir=True: L bidirectional blocks. bidir=False: 2L forward-only blocks (same parameter count), causal over the sequence."""
    def __init__(self, d, L, bidir=True):
        super().__init__()
        self.blocks = nn.ModuleList([BiSSMBlock(d, bidir) for _ in range(L if bidir else 2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

def sinusoid(n, d, device):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
    return pe

class TransformerMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(d, CFG["N_HEADS_ATTN"], 4 * d, dropout=0.1, activation="gelu",
                                                                batch_first=True, norm_first=True) for _ in range(L)])
    def forward(self, h, m):
        h = h + sinusoid(h.size(1), h.size(2), h.device).to(h.dtype)
        kpm = ~m
        for layer in self.layers:
            h = run_block(layer, h, None, kpm)
        return h

class GRUMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__()
        self.rnn = nn.GRU(d, d, L, batch_first=True, bidirectional=True, dropout=0.1)
        self.proj = nn.Linear(2 * d, d)
    def forward(self, h, m):
        return self.proj(run_block(self.rnn, h * m[..., None])[0])

class ConvBlock(nn.Module):
    def __init__(self, d, dil):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.dw = nn.Conv1d(d, d, 9, padding=4 * dil, dilation=dil, groups=d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop = nn.Dropout(CFG["DROPOUT"])
    def forward(self, h, m):
        x = (self.norm(h) * m[..., None]).transpose(1, 2)
        return h + self.drop(self.mlp(self.dw(x).transpose(1, 2)))

class CNNMixer(nn.Module):
    def __init__(self, d, L):
        super().__init__(); self.blocks = nn.ModuleList([ConvBlock(d, 2 ** (i % 6)) for i in range(2 * L)])
    def forward(self, h, m):
        for blk in self.blocks:
            h = run_block(blk, h, m)
        return h

MIXERS = dict(mamba=SSMMixer, mamba_causal=lambda d, L: SSMMixer(d, L, bidir=False),
              transformer=TransformerMixer, gru=GRUMixer, cnn=CNNMixer)

class Stem(nn.Module):
    def __init__(self, d, patch):
        super().__init__()
        self.patch = patch
        self.c1 = nn.Conv1d(N_CH, d, 7, padding=3)
        self.c2 = nn.Conv1d(d, d, patch, stride=patch)
        self.norm = nn.LayerNorm(d)
    def forward(self, x, m):
        h = self.c2(F.gelu(self.c1(x * m[:, None]))).transpose(1, 2)
        tm = F.avg_pool1d(m[:, None], self.patch, self.patch)[:, 0] > 0.5
        return self.norm(h) * tm[..., None], tm

class Net(nn.Module):
    def __init__(self, mixer):
        super().__init__()
        d = CFG["D_MODEL"]
        self.stem = Stem(d, CFG["PATCH"])
        self.mixer = MIXERS[mixer](d, CFG["N_LAYERS"])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, len(CLASSES)))
        self.half_center = int(5 * CFG["FS"] / CFG["PATCH"])     # tokens in 5 s

    def forward(self, x, m):
        h, tm = self.stem(x, m)
        h = self.norm(self.mixer(h, tm))
        c = h.size(1) // 2
        cm = torch.zeros_like(tm)
        cm[:, max(c - self.half_center, 0):c + self.half_center] = True
        cm &= tm
        def pool(mask):
            w = mask.to(h.dtype)[..., None]
            return (h * w).sum(1) / w.sum(1).clamp(min=1)
        return self.head(torch.cat([pool(cm), pool(tm)], dim=-1))

def n_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## v11 settings

In [7]:
CFG.update(dict(
    CACHE_CTX_SEC=200, CACHE_DIR="/tmp/hms_cache", RESULTS_DIR="/kaggle/working/results_v11",
    PREV_GLOBS=["/kaggle/input/**/results_v11/*.json", "/kaggle/input/**/results_v11/*_valpred.npy", "/kaggle/input/**/results_v11/*.csv"],
    PRED_GLOBS=["/kaggle/input/**/*_valpred.npy"], CKPT_GLOBS=["/kaggle/input/**/*.pt"],
    CTX=200, N_FOLDS=5, EPOCHS=6, BATCH=64, LR=1e-3, WD=0.05, WARMUP_FRAC=0.05, CLIP=1.0, TOKENS_PER_BATCH=25600,
    DROPOUT=0.1, GRAD_CKPT=False, USE_OFFICIAL_MAMBA2=False, P_TRUNC=0.5, EVAL_BATCH=64,
    RUN_PROBE=True, PROBE_MODELS=["mamba", "gru"], PROBE_C=1.0,
    RUNS=[("mamba", "ba_splice"), ("gru", "ba_splice"), ("mamba", "ba_noise"), ("gru", "ba_noise")],
    EST_MIN_PER_FOLD=dict(mamba=17.5, gru=4.2), TIME_BUDGET_H=6.0,
    B=4000, MIN_N=20, T_GRID=list(np.round(np.linspace(0.5, 3.0, 51), 3)),
    CTX_BINS=[-1, 10, 140, 1000], CTX_LABELS=["none (<10 s)", "partial (10-140 s)", "full (>=140 s)"],
    VOTE_GROUPS=[(1, 2, "1-2 votes"), (3, 9, "3-9 votes"), (10, 10_000, ">=10 votes")],
))
CFG_OVERRIDES = {}
CFG.update(CFG_OVERRIDES)
T_START = time.time()
os.makedirs(CFG["CACHE_DIR"], exist_ok=True); os.makedirs(CFG["RESULTS_DIR"], exist_ok=True)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)

n_copied = 0
for pat in CFG["PREV_GLOBS"]:
    for fp in glob.glob(pat, recursive=True):
        dst = os.path.join(CFG["RESULTS_DIR"], os.path.basename(fp))
        if not os.path.exists(dst):
            shutil.copy(fp, dst); n_copied += 1
print(f"resumed {n_copied} files from earlier v11 outputs")
def find_files(globs):
    out = {}
    for pat in globs:
        for fp in sorted(glob.glob(pat, recursive=True)):
            out.setdefault(os.path.basename(fp), fp)
    return out
PREDS, CKPTS = find_files(CFG["PRED_GLOBS"]), find_files(CFG["CKPT_GLOBS"])
print(f"found {len(PREDS)} prediction files and {len(CKPTS)} checkpoints")

def run_tag(name, variant, fold):
    return f"{name}_{variant}_ctx{CFG['CTX']}_f{fold}"

resumed 0 files from earlier v11 outputs
found 260 prediction files and 135 checkpoints


## Per-EEG and per-row context availability

In [8]:
cache_idx = build_cache(df)
CACHE = {int(r.eeg_id): (int(r.offset), int(r.s0), int(r.length)) for r in cache_idx.itertuples()}
DUR = dict(zip(cache_idx["eeg_id"].astype(int), cache_idx["n_src"] / SRC_FS))

def extra_context(center, dur, ctx=CFG["CTX"]):
    half = ctx / 2
    left = np.minimum(center, half); right = np.clip(np.minimum(dur - center, half), 0, None)
    return left + right - np.minimum(center, 25) - np.clip(np.minimum(dur - center, 25), 0, None)

df["dur_s"] = df["eeg_id"].map(DUR)
df["extra_s"] = extra_context(df["center"].to_numpy(), df["dur_s"].to_numpy())
df["ctx_bin"] = pd.cut(df["extra_s"], CFG["CTX_BINS"], labels=CFG["CTX_LABELS"]).astype(str)

VA = {f: middle_row_per_eeg(df[df.fold == f]).reset_index(drop=True) for f in range(CFG["N_FOLDS"])}
OOF = pd.concat([VA[f] for f in range(CFG["N_FOLDS"])], ignore_index=True)
T = OOF[P_COLS].to_numpy(np.float64)
NV = OOF["n_votes"].to_numpy(); PID = OOF["patient_id"].to_numpy(); FOLD = OOF["fold"].to_numpy()
OOF["vote_group"] = ""
for lo, hi, name in CFG["VOTE_GROUPS"]:
    OOF.loc[(NV >= lo) & (NV <= hi), "vote_group"] = name
OOF["label"] = np.array(CLASSES)[OOF["y"].to_numpy()]
VG, CB = OOF["vote_group"].to_numpy(), OOF["ctx_bin"].to_numpy()
print(pd.crosstab(OOF["vote_group"], OOF["ctx_bin"], margins=True).to_string())

def per_eeg_kl(P, eps=1e-15):
    return (T * (np.log(T + eps) - np.log(P + eps))).sum(1)

def load_oof(model, ctx, stage):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        fp = PREDS.get(f"{model}_ctx{ctx}_f{f}_{stage}_valpred.npy")
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

UP, INV = np.unique(PID, return_inverse=True)
W = np.random.default_rng(CFG["SEED"]).multinomial(len(UP), np.full(len(UP), 1.0 / len(UP)), size=CFG["B"]).astype(np.float32)

def boot(d, msk):
    if msk.sum() == 0:
        return (float("nan"),) * 4
    s_ = np.bincount(INV[msk], weights=d[msk], minlength=len(UP)).astype(np.float32)
    n_ = np.bincount(INV[msk], minlength=len(UP)).astype(np.float32)
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (W @ s_) / (W @ n_)
    r = r[np.isfinite(r)]
    if len(r) == 0:
        return float(d[msk].mean()), float("nan"), float("nan"), float("nan")
    lo, hi = np.percentile(r, [2.5, 97.5])
    return float(d[msk].mean()), lo, hi, max(min(1.0, 2 * min((r <= 0).mean(), (r >= 0).mean())), 1.0 / len(r))

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = np.isfinite(p)
    idx = np.argsort(np.where(ok, p, np.inf)); k = ok.sum(); run = 0.0
    for i_, j in enumerate(idx[:k]):
        run = max(run, min(1.0, (k - i_) * p[j])); out[j] = run
    return out

SUBSETS = {"all EEGs": np.ones(len(OOF), bool)}
for b in CFG["CTX_LABELS"]:
    SUBSETS[f"all | {b}"] = CB == b
for g in ["1-2 votes", "3-9 votes", ">=10 votes"]:
    SUBSETS[f"{g} | all"] = VG == g
    for b in CFG["CTX_LABELS"]:
        SUBSETS[f"{g} | {b}"] = (VG == g) & (CB == b)

cache: 380.3 h of EEG, 4.93 GB needed, 1038.2 GB free
  0/17,089  1s
  2,000/17,089  21s
  4,000/17,089  41s
  6,000/17,089  61s
  8,000/17,089  81s
  10,000/17,089  102s
  12,000/17,089  121s
  14,000/17,089  141s
  16,000/17,089  161s
cache built in 174s
ctx_bin     full (>=140 s)  none (<10 s)  partial (10-140 s)    All
vote_group                                                         
1-2 votes               25          4526                 296   4847
3-9 votes              297          2298                4047   6642
>=10 votes             169          2648                2783   5600
All                    491          9472                7126  17089


## Datasets: standard view (zeros + mask), BA view with mirror / splice / noise fill

In [9]:
SOS100 = butter(4, [0.5, 40.0], btype="band", fs=CFG["FS"], output="sos")

class FillDataset(Dataset):
    """200 s (or ctx) window around the label centre.
    view "standard": zeros + mask 0 outside the recording (original models). view "ba": fill outside the real signal, mask 1;
    view "trunc50": only the central 50 s real, the rest filled. Training (train=True, view "ba") also hides real context at random (p = P_TRUNC)."""
    def __init__(self, rows, ctx_sec, train, view="ba", fill="mirror"):
        self.eid = rows["eeg_id"].to_numpy(); self.center = rows["center"].to_numpy(np.float64)
        self.p = rows[P_COLS].to_numpy(np.float32)
        self.fs, self.src_fs, self.n_ch = CFG["FS"], SRC_FS, N_CH
        self.n = int(ctx_sec * self.fs); self.train, self.view, self.fill = train, view, fill
        self.jitter, self.p_trunc, self.flip = CFG["JITTER_SEC"], CFG["P_TRUNC"], np.array(FLIP_PERM)
        self.cache = {int(e): CACHE[int(e)] for e in np.unique(self.eid)}
        lens = np.array([self.cache[int(e)][2] for e in np.unique(self.eid)]); ids = np.unique(self.eid)
        o = np.argsort(lens); self.donor_ids, self.donor_lens = ids[o], lens[o]
        self.path = f"{CFG['CACHE_DIR']}/signals.npy"; self.mm = None
    def __len__(self):
        return len(self.eid)
    def donor(self, L, avoid, rng):
        k = np.searchsorted(self.donor_lens, L)
        pool = self.donor_ids[k:] if k < len(self.donor_ids) else self.donor_ids[-1:]
        e = int(pool[rng.integers(len(pool))])
        if e == avoid and len(pool) > 1:
            e = int(pool[(list(pool).index(e) + 1) % len(pool)])
        off, _, length = self.cache[e]
        if length >= L:
            st = int(rng.integers(0, length - L + 1))
            return np.asarray(self.mm[off + st:off + st + L], dtype=np.float32)
        seg = np.asarray(self.mm[off:off + length], dtype=np.float32)
        return np.resize(seg, (L, self.n_ch)) if length > 0 else np.zeros((L, self.n_ch), np.float32)
    def filled(self, real, lo, hi, avoid, rng):
        if self.fill == "mirror":
            return np.pad(real, ((lo, self.n - hi), (0, 0)), mode="symmetric")
        x = np.zeros((self.n, self.n_ch), np.float32); x[lo:hi] = real
        sd = real.std(0, keepdims=True) + 1e-6
        for a, b in ((0, lo), (hi, self.n)):
            L = b - a
            if L <= 0:
                continue
            if self.fill == "splice":
                x[a:b] = self.donor(L, avoid, rng)
            elif self.fill == "noise":
                z = sosfiltfilt(SOS100, rng.standard_normal((L + 400, self.n_ch)), axis=0)[200:200 + L]
                x[a:b] = z / (z.std(0, keepdims=True) + 1e-6) * sd
        return x
    def __getitem__(self, i):
        if self.mm is None:
            self.mm = np.load(self.path, mmap_mode="r")
        rng = np.random.default_rng() if self.train else np.random.default_rng(1_000_003 * i + 17)
        off, s0, length = self.cache[int(self.eid[i])]
        c = self.center[i] + (rng.uniform(-self.jitter, self.jitter) if self.train else 0.0)
        i0 = int(round((c - self.n / self.fs / 2 - s0 / self.src_fs) * self.fs))
        a, b = max(i0, 0), min(i0 + self.n, length)
        lo, hi = a - i0, b - i0
        m = np.ones(self.n, np.float32)
        if self.view == "standard":
            x = np.zeros((self.n, self.n_ch), np.float32); m = np.zeros(self.n, np.float32)
            if hi > lo:
                x[lo:hi] = self.mm[off + a:off + b]; m[lo:hi] = 1.0
        else:
            cidx, h50 = self.n // 2, 25 * self.fs
            if self.view == "trunc50":
                lo, hi = max(lo, cidx - h50), min(hi, cidx + h50)
            elif self.train and rng.random() < self.p_trunc:
                lo = int(rng.integers(lo, max(lo, cidx - h50) + 1))
                hi = int(rng.integers(min(hi, cidx + h50), hi + 1))
            if hi <= lo:
                x = np.zeros((self.n, self.n_ch), np.float32); m[:] = 0.0
            else:
                real = np.asarray(self.mm[off + i0 + lo:off + i0 + hi], dtype=np.float32)
                x = real if (lo == 0 and hi == self.n) else self.filled(real, lo, hi, int(self.eid[i]), rng)
        x = x.T / 50.0
        if self.train:
            if rng.random() < 0.5:
                x = x[self.flip]
            x = x * rng.uniform(0.9, 1.1)
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)), torch.from_numpy(m),
                torch.from_numpy(self.p[i]), torch.tensor(1.0))

# sanity: the central 50 s is identical across fills; outside it, splice/noise differ from mirror
_r = OOF.head(32)
_v = {fl: FillDataset(_r, CFG["CTX"], False, "trunc50", fl) for fl in ("mirror", "splice", "noise")}
for i in range(len(_r)):
    xs = {fl: _v[fl][i][0].numpy() for fl in _v}
    c0, c1 = CFG["CTX"] * CFG["FS"] // 2 - 25 * CFG["FS"], CFG["CTX"] * CFG["FS"] // 2 + 25 * CFG["FS"]
    assert np.allclose(xs["mirror"][:, c0:c1], xs["splice"][:, c0:c1]) and np.allclose(xs["mirror"][:, c0:c1], xs["noise"][:, c0:c1])
    assert np.isfinite(xs["noise"]).all() and np.isfinite(xs["splice"]).all()
print("fill sanity checks passed")

def param_groups(model):
    decay, no_decay = [], []
    for n_, p in model.named_parameters():
        (no_decay if p.ndim < 2 or n_.endswith(("A_log", "D", "dt_bias")) else decay).append(p)
    return [dict(params=decay, weight_decay=CFG["WD"]), dict(params=no_decay, weight_decay=0.0)]

def loader(ds, batch, sampler=None, drop_last=False):
    return DataLoader(ds, batch, sampler=sampler, shuffle=False, drop_last=drop_last, num_workers=CFG["NUM_WORKERS"],
                      pin_memory=DEV.type == "cuda", persistent_workers=CFG["NUM_WORKERS"] > 0,
                      multiprocessing_context=MP if CFG["NUM_WORKERS"] > 0 else None)

@torch.no_grad()
def predict(model, dl, embed=False):
    model.eval(); out, emb = [], []
    hook = model.head.register_forward_pre_hook(lambda mod, args: emb.append(args[0].float().cpu())) if embed else None
    for x, m, _, _ in dl:
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=CFG["AMP"] and DEV.type == "cuda"):
            out.append(torch.softmax(model(x.to(DEV), m.to(DEV)).float(), -1).cpu())
    if hook is not None:
        hook.remove()
    P = torch.cat(out).numpy()
    return (P, torch.cat(emb).numpy()) if embed else P

fill sanity checks passed


## A. Embedding probe: is "short excerpt" linearly decodable from the representation?

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold

NOCTX = (CB == CFG["CTX_LABELS"][0])
PROBE_SETS = [("50 s", 50, "standard", "mirror", "{m}_ctx50_f{f}_s1.pt"),
              ("200 s original", 200, "standard", "mirror", "{m}_ctx200_f{f}_s1.pt"),
              ("200 s BA (mirror)", 200, "ba", "mirror", "{m}_ba_noreweight_ctx200_f{f}.pt")]

def probe_auc(X, y, groups):
    """patient-grouped 5-fold CV logistic probe; returns out-of-sample AUC (nan if one class only)"""
    if y.min() == y.max() or len(np.unique(groups)) < 5:
        return np.nan
    pr = np.zeros(len(y))
    for tr_, te_ in GroupKFold(5).split(X, y, groups):
        if y[tr_].min() == y[tr_].max():
            pr[te_] = y[tr_].mean(); continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=CFG["PROBE_C"], max_iter=3000))
        clf.fit(X[tr_], y[tr_]); pr[te_] = clf.predict_proba(X[te_])[:, 1]
    return roc_auc_score(y, pr)

probe_csv = f"{CFG['RESULTS_DIR']}/v11_probe.csv"
if CFG["RUN_PROBE"] and not os.path.exists(probe_csv):
    rows = []
    for f in range(CFG["N_FOLDS"]):
        va = VA[f]; idx = np.where(FOLD == f)[0]
        y, g, hv = NOCTX[idx].astype(int), PID[idx], NV[idx] >= 10
        rows.append(dict(model="(labels)", setting="true label vector", fold=f, auc_all=probe_auc(T[idx], y, g), auc_ge10=probe_auc(T[idx][hv], y[hv], g[hv])))
        for mname in CFG["PROBE_MODELS"]:
            for sname, ctx, view, fill, pat in PROBE_SETS:
                ck = CKPTS.get(pat.format(m=mname, f=f))
                if ck is None:
                    print(f"  missing checkpoint {pat.format(m=mname, f=f)}: skipped"); continue
                model = Net(mname).to(DEV); model.load_state_dict(torch.load(ck, map_location=DEV))
                P, E = predict(model, loader(FillDataset(va, ctx, False, view, fill), CFG["EVAL_BATCH"]), embed=True)
                rows.append(dict(model=mname, setting=sname, fold=f, auc_all=probe_auc(E, y, g), auc_ge10=probe_auc(E[hv], y[hv], g[hv]),
                                 auc_from_outputs=probe_auc(np.log(np.clip(P, 1e-8, 1)), y, g)))
                print(f"  fold {f} {mname:6s} {sname:18s} probe AUC all {rows[-1]['auc_all']:.3f} | >=10 votes {rows[-1]['auc_ge10']:.3f}")
                del model
                if DEV.type == "cuda": torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(probe_csv, index=False)
if os.path.exists(probe_csv):
    pr_ = pd.read_csv(probe_csv)
    summ = pr_.groupby(["model", "setting"], sort=False).agg(folds=("fold", "size"), auc_all=("auc_all", "mean"), auc_all_sd=("auc_all", "std"),
                                                            auc_ge10=("auc_ge10", "mean"), auc_ge10_sd=("auc_ge10", "std"),
                                                            auc_from_outputs=("auc_from_outputs", "mean")).reset_index()
    summ.to_csv(f"{CFG['RESULTS_DIR']}/v11_probe_summary.csv", index=False)
    print("### Probe AUC for 'no extra real context' (mean over the 5 fold models)\n" + summ.round(3).to_string(index=False))

  fold 0 mamba  50 s               probe AUC all 0.671 | >=10 votes 0.594
  fold 0 mamba  200 s original     probe AUC all 0.961 | >=10 votes 0.935
  fold 0 mamba  200 s BA (mirror)  probe AUC all 0.674 | >=10 votes 0.625
  fold 0 gru    50 s               probe AUC all 0.695 | >=10 votes 0.570
  fold 0 gru    200 s original     probe AUC all 0.913 | >=10 votes 0.831
  fold 0 gru    200 s BA (mirror)  probe AUC all 0.673 | >=10 votes 0.558
  fold 1 mamba  50 s               probe AUC all 0.585 | >=10 votes 0.613
  fold 1 mamba  200 s original     probe AUC all 0.949 | >=10 votes 0.906
  fold 1 mamba  200 s BA (mirror)  probe AUC all 0.601 | >=10 votes 0.591
  fold 1 gru    50 s               probe AUC all 0.599 | >=10 votes 0.603
  fold 1 gru    200 s original     probe AUC all 0.924 | >=10 votes 0.893
  fold 1 gru    200 s BA (mirror)  probe AUC all 0.594 | >=10 votes 0.619
  fold 2 mamba  50 s               probe AUC all 0.644 | >=10 votes 0.565
  fold 2 mamba  200 s original     pro

## B. Fill-type ablation: BA training with splice or noise fill

In [11]:
def train_fill(name, variant, fold):
    tag = run_tag(name, variant, fold); fill = variant.split("_")[1]
    seed_all(CFG["SEED"] + fold)
    tr = df[df.fold != fold].reset_index(drop=True)
    tok = CFG["CTX"] * CFG["FS"] // CFG["PATCH"]
    micro = max(1, min(CFG["BATCH"], CFG["TOKENS_PER_BATCH"] // tok)); accum = math.ceil(CFG["BATCH"] / micro)
    dl = loader(FillDataset(tr, CFG["CTX"], True, "ba", fill), micro, OnePerEEG(tr["eeg_id"].to_numpy(), CFG["SEED"] + fold), drop_last=True)
    model = Net(name).to(DEV)
    opt = torch.optim.AdamW(param_groups(model), lr=CFG["LR"])
    total = max(1, CFG["EPOCHS"] * (len(dl) // accum)); warm = max(1, int(CFG["WARMUP_FRAC"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    use_amp = CFG["AMP"] and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"\n=== {tag} | fill {fill} | micro {micro} x accum {accum}")
    t_run, hist = time.time(), []
    for ep in range(CFG["EPOCHS"]):
        model.train(); t0 = time.time(); tl = nb = 0.0
        opt.zero_grad(set_to_none=True)
        for i, (x, m, p, _) in enumerate(dl):
            x, m, p = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, m)
            loss = F.kl_div(F.log_softmax(logits.float(), -1), p, reduction="batchmean")
            scaler.scale(loss / accum).backward()
            tl += loss.item(); nb += 1
            if (i + 1) % accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), sec=time.time() - t0))
        print(f"  {tag} ep {ep} | train KL {tl/max(nb,1):.4f} | {time.time()-t0:.0f}s")
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, f"{CFG['RESULTS_DIR']}/{tag}.pt")
    for view in ("ba", "trunc50"):
        np.save(f"{CFG['RESULTS_DIR']}/{tag}_{view}_valpred.npy", predict(model, loader(FillDataset(VA[fold], CFG["CTX"], False, view, fill), micro * 2)))
    json.dump(dict(model=name, variant=variant, fold=fold, minutes=(time.time() - t_run) / 60, history=hist),
              open(f"{CFG['RESULTS_DIR']}/{tag}.json", "w"), indent=1)
    print(f"=== {tag} done in {(time.time()-t_run)/60:.1f} min")
    del model
    if DEV.type == "cuda": torch.cuda.empty_cache()

def done(name, variant, fold):
    return all(os.path.exists(f"{CFG['RESULTS_DIR']}/{run_tag(name, variant, fold)}_{v}_valpred.npy") for v in ("ba", "trunc50"))

PLAN = [(n_, v_, f_) for n_, v_ in CFG["RUNS"] for f_ in range(CFG["N_FOLDS"])]
todo = [r for r in PLAN if not done(*r)]
print(f"{len(PLAN)} runs planned, {len(PLAN) - len(todo)} already done, ~{sum(CFG['EST_MIN_PER_FOLD'].get(r[0], 17) for r in todo) / 60:.1f} GPU-h left")
for n_, v_, f_ in todo:
    used = (time.time() - T_START) / 3600
    if used + CFG["EST_MIN_PER_FOLD"].get(n_, 17) / 60 > CFG["TIME_BUDGET_H"]:
        print(f"time budget: stopping before {run_tag(n_, v_, f_)} ({used:.2f} h used). Attach this output and rerun to continue.")
        break
    train_fill(n_, v_, f_)

20 runs planned, 0 already done, ~3.6 GPU-h left

=== mamba_ba_splice_ctx200_f0 | fill splice | micro 32 x accum 2
  mamba_ba_splice_ctx200_f0 ep 0 | train KL 1.0121 | 169s
  mamba_ba_splice_ctx200_f0 ep 1 | train KL 0.7456 | 166s
  mamba_ba_splice_ctx200_f0 ep 2 | train KL 0.6607 | 165s
  mamba_ba_splice_ctx200_f0 ep 3 | train KL 0.5938 | 165s
  mamba_ba_splice_ctx200_f0 ep 4 | train KL 0.5453 | 166s
  mamba_ba_splice_ctx200_f0 ep 5 | train KL 0.5160 | 166s
=== mamba_ba_splice_ctx200_f0 done in 17.0 min

=== mamba_ba_splice_ctx200_f1 | fill splice | micro 32 x accum 2
  mamba_ba_splice_ctx200_f1 ep 0 | train KL 1.0062 | 156s
  mamba_ba_splice_ctx200_f1 ep 1 | train KL 0.7406 | 155s
  mamba_ba_splice_ctx200_f1 ep 2 | train KL 0.6486 | 155s
  mamba_ba_splice_ctx200_f1 ep 3 | train KL 0.5871 | 155s
  mamba_ba_splice_ctx200_f1 ep 4 | train KL 0.5386 | 156s
  mamba_ba_splice_ctx200_f1 ep 5 | train KL 0.5114 | 156s
=== mamba_ba_splice_ctx200_f1 done in 16.1 min

=== mamba_ba_splice_ctx200_f

## Results: does the fill type change the residual shortcut or the real-context effect?
Calibrated (cross-fitted temperature fitted on the model's full `ba` view and reused for `trunc50`); patient-level bootstrap. Reference: the 50 s model of the same family (seed 42). Mirror = the existing BA no-reweight models (seed 42).

In [12]:
def pooled(fmt):
    parts = []
    for f in range(CFG["N_FOLDS"]):
        b = fmt.format(f=f); fp = f"{CFG['RESULTS_DIR']}/{b}" if os.path.exists(f"{CFG['RESULTS_DIR']}/{b}") else PREDS.get(b)
        if fp is None:
            return None
        P = np.load(fp); assert len(P) == len(VA[f]), fp
        parts.append(P)
    return np.concatenate(parts).astype(np.float64)

def temper(P, t):
    L_ = np.log(np.clip(P, 1e-12, 1)) / t; L_ -= L_.max(1, keepdims=True); E = np.exp(L_)
    return E / E.sum(1, keepdims=True)
def fit_T(P):
    return {f: min(CFG["T_GRID"], key=lambda t: per_eeg_kl(temper(P, t))[FOLD != f].mean()) for f in range(CFG["N_FOLDS"])}
def apply_T(P, Ts):
    out = np.empty_like(P)
    for f, t in Ts.items():
        out[FOLD == f] = temper(P[FOLD == f], t)
    return out

ALL = np.ones(len(OOF), bool); NONE, FULL = CB == CFG["CTX_LABELS"][0], CB == CFG["CTX_LABELS"][2]; REAL = ~NONE
def rel_reps(ka, kb, msk):
    s_a = np.bincount(INV[msk], weights=(ka - kb)[msk], minlength=len(UP)).astype(np.float32)
    s_b = np.bincount(INV[msk], weights=kb[msk], minlength=len(UP)).astype(np.float32)
    return (W @ s_a) / np.maximum(W @ s_b, 1e-9), (ka - kb)[msk].mean() / kb[msk].mean()

rows, arows = [], []
for name in ["mamba", "gru"]:
    P50 = pooled(f"{name}_ctx50_f{{f}}_s1_valpred.npy")
    if P50 is None:
        print(f"{name}: 50 s reference predictions missing"); continue
    k50 = per_eeg_kl(apply_T(P50, fit_T(P50)))
    for fill, variant in [("mirror", "ba_noreweight"), ("splice", "ba_splice"), ("noise", "ba_noise")]:
        Pb, Pt = pooled(f"{name}_{variant}_ctx200_f{{f}}_ba_valpred.npy"), pooled(f"{name}_{variant}_ctx200_f{{f}}_trunc50_valpred.npy")
        if Pb is None or Pt is None:
            print(f"{name} {fill}: predictions missing"); continue
        Ts = fit_T(Pb); kb, kt = per_eeg_kl(apply_T(Pb, Ts)), per_eeg_kl(apply_T(Pt, Ts))
        rows.append(dict(model=name, fill=fill, kl_all=kb.mean(), kl_ge10=kb[NV >= 10].mean(), kl_50s=k50.mean()))
        rn, pn = rel_reps(kb, k50, NONE); rf, pf = rel_reps(kt, k50, FULL); r = rn - rf
        lo, hi = np.percentile(r, [2.5, 97.5]); p = max(min(1.0, 2 * min((r <= 0).mean(), (r >= 0).mean())), 1.0 / len(r))
        m_, clo, chi, cp = boot(kb - kt, REAL)
        sh = (apply_T(Pt, Ts)[FULL] - apply_T(Pb, Ts)[FULL]).mean(0)
        arows.append(dict(model=name, fill=fill, gain_none_pct=100 * pn, gain_full_trunc_pct=100 * pf, asymmetry_pts=100 * (pn - pf),
                          ci_lo=100 * lo, ci_hi=100 * hi, p=p, real_context_rel_pct=100 * m_ / kt[REAL].mean(), real_ci_lo=100 * clo / kt[REAL].mean(),
                          real_ci_hi=100 * chi / kt[REAL].mean(), real_p=cp, max_abs_trunc_shift=np.abs(sh).max(), shift_Seizure=sh[0], shift_Other=sh[5]))
t_kl = pd.DataFrame(rows); t_as = pd.DataFrame(arows)
if len(t_as):
    t_as["p_holm"] = holm(t_as["p"].to_numpy()); t_as["real_p_holm"] = holm(t_as["real_p"].to_numpy())
t_kl.to_csv(f"{CFG['RESULTS_DIR']}/v11_fill_kl.csv", index=False); t_as.to_csv(f"{CFG['RESULTS_DIR']}/v11_fill_asymmetry.csv", index=False)
print("### Calibrated KL by fill\n" + t_kl.round(4).to_string(index=False))
print("\n### Matched-input asymmetry (points; 0 = no shortcut) and real-context effect (full BA view vs central 50 s, EEGs with real context)\n"
      + t_as.round(3).to_string(index=False))
print("outputs:", len(os.listdir(CFG["RESULTS_DIR"])), "files")

### Calibrated KL by fill
model   fill  kl_all  kl_ge10  kl_50s
mamba mirror  0.6727   0.5157  0.7001
mamba splice  0.7035   0.5463  0.7001
mamba  noise  0.6670   0.5573  0.7001
  gru mirror  0.6832   0.5285  0.7066
  gru splice  0.7022   0.5505  0.7066
  gru  noise  0.6660   0.5646  0.7066

### Matched-input asymmetry (points; 0 = no shortcut) and real-context effect (full BA view vs central 50 s, EEGs with real context)
model   fill  gain_none_pct  gain_full_trunc_pct  asymmetry_pts   ci_lo   ci_hi     p  real_context_rel_pct  real_ci_lo  real_ci_hi  real_p  max_abs_trunc_shift  shift_Seizure  shift_Other  p_holm  real_p_holm
mamba mirror         -3.651               -5.282          1.631  -3.098   6.400 0.483                -3.459      -4.400      -2.506     0.0                0.016         -0.016        0.016   1.000        0.002
mamba splice          0.119                1.104         -0.985  -6.135   4.000 0.694                -2.929      -3.636      -2.253     0.0               